# 01 — Data exploration (ART)

Schema, label balance, length statistics after preprocessing, OOV rates and a few examples. All numbers below are computed live from the dataset.

In [1]:
import sys
sys.path.append('../src')  # make src/ importable

from collections import Counter
import pandas as pd
from data import load_art, tokenize, get_vocab_and_embeddings, oov_rate, length_stats, label_to_index

## Schema

In [2]:
ds = load_art()
print(ds)
ds['train'].features

DatasetDict({
    train: Dataset({
        features: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label'],
        num_rows: 16801
    })
    validation: Dataset({
        features: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label'],
        num_rows: 1532
    })
})


{'observation_1': Value('string'),
 'observation_2': Value('string'),
 'hypothesis_1': Value('string'),
 'hypothesis_2': Value('string'),
 'label': ClassLabel(names=['0', '1', '2'])}

## Label distribution

Labels are 1/2 (index of the correct hypothesis). We map them to 0/1. A balanced distribution means the majority-class baseline is close to 50%.

In [3]:
rows = []
for name, split in ds.items():
    c = Counter(split['label'])
    rows.append({'split': name, 'n': len(split), 'label=1': c[1], 'label=2': c[2], 'share label=1': c[1] / len(split)})
pd.DataFrame(rows)

,split,n,label=1,label=2,share label=1
0,train,16801,8461,8340,0.503601
1,validation,1000,508,492,0.508000
2,test,1532,781,751,0.509791


## A few examples (train)

In [4]:
for ex in ds['train'].select(range(3)):
    print('O1:', ex['observation_1'])
    print('H1:', ex['hypothesis_1'])
    print('H2:', ex['hypothesis_2'])
    print('O2:', ex['observation_2'])
    print('label:', ex['label'], '-> index', label_to_index(ex['label']))
    print('tokens H1:', tokenize(ex['hypothesis_1']))
    print()

O1: A baby panda was lost in the woods.
H1: He was never lost.
H2: All the people from the zoo looked for him.
O2: He was reunited with his parents, who also lived at the zoo.
label: 2 -> index 1
tokens H1: ['he', 'was', 'never', 'lost']

O1: A bank teller told me to rob the bank where she worked.
H1: The mountain told me when no one would be there.
H2: The teller told me when no one would be there.
O2: We split the money equally.
label: 2 -> index 1
tokens H1: ['the', 'mountain', 'told', 'me', 'when', 'no', 'one', 'would', 'be', 'there']

O1: A bee was buzzing around my head.
H1: It was because of the honey in my hair.
H2: It was because of the blood in my stool.
O2: I didn't bother to explain the situation.
label: 1 -> index 0
tokens H1: ['it', 'was', 'because', 'of', 'the', 'honey', 'in', 'my', 'hair']



## Token lengths after preprocessing

Used to justify *no truncation*: even the longest RNN input (`O1 <sep> H <sep> O2`) is short.

In [5]:
pd.concat({name: pd.DataFrame(length_stats(split)).T for name, split in ds.items()})

mean  median   p95   max   min
train      observation_1   8.191239     8.0  13.0  17.0   2.0
           observation_2   9.384084    10.0  14.0  19.0   2.0
           hypothesis_1    8.510446     8.0  14.0  30.0   0.0
           hypothesis_2    8.531159     8.0  14.0  30.0   0.0
           rnn_sequence   28.096125    28.0  37.0  58.0  11.0
validation observation_1   8.437000     8.0  13.0  16.0   3.0
           observation_2   9.436000     9.0  14.0  17.0   2.0
           hypothesis_1    8.597000     8.0  14.0  31.0   3.0
           hypothesis_2    8.610000     8.0  14.0  31.0   3.0
           rnn_sequence   28.476500    28.0  37.0  52.0  14.0
test       observation_1   8.168407     8.0  13.0  17.0   3.0
           observation_2   9.408616    10.0  14.0  17.0   2.0
           hypothesis_1    8.723238     8.0  15.0  27.0   2.0
           hypothesis_2    8.740209     8.0  15.0  24.0   2.0
           rnn_sequence   28.308747    28.0  37.0  51.0  14.0

## Empty texts after preprocessing

A handful of training hypotheses are placeholders like `'{}.'` and become empty. They are kept; the models handle empty texts (zero mean vector / RNN still sees the two `<sep>` tokens).

In [6]:
empty = [ex for ex in ds['train'] if not tokenize(ex['hypothesis_1']) or not tokenize(ex['hypothesis_2'])]
print(len(empty))
pd.DataFrame(empty)[['hypothesis_1', 'hypothesis_2', 'label']]

6


,hypothesis_1,hypothesis_2,label
0,Drew went to jail for theft.,{}.,1
1,{}.,I went to the concert.,2
2,Amy ran in to explain what had happened.,.,1
3,{}.,Vianna graduated at the top of her class.,2
4,.,Warner had a bittersweet night of gambling.,2
5,I had taken to much cold medicine.,{}.,1


## Vocabulary and OOV rate

Vocabulary = train tokens that have a GloVe vector (+ `<pad>`, `<unk>`, `<sep>`). Train OOV only counts train words missing from GloVe; validation/test OOV also counts words never seen in training.

In [7]:
itos, embeddings = get_vocab_and_embeddings(ds['train'], cache_dir='../cache')
stoi = {w: i for i, w in enumerate(itos)}
print('vocab size:', len(itos), '| embedding matrix:', tuple(embeddings.shape))
pd.DataFrame({'OOV rate': {name: oov_rate(split, stoi) for name, split in ds.items()}})

vocab size: 15437 | embedding matrix: (15437, 300)


,OOV rate
train,0.001900
validation,0.017503
test,0.017790


In [8]:
# Most frequent OOV tokens in the validation split
oov = Counter(t for ex in ds['validation'] for f in ('observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2')
              for t in tokenize(ex[f]) if t not in stoi)
oov.most_common(20)

[('yu', 6),
 ('darvish', 6),
 ('wally', 6),
 ('willy', 6),
 ('yaris', 5),
 ('trey', 5),
 ('weston', 5),
 ('jerald', 4),
 ('lanie', 4),
 ('macie', 4),
 ('miki', 4),
 ('espresso', 4),
 ('asus', 4),
 ('brant', 4),
 ('yusuf', 4),
 ('usman', 4),
 ('wendel', 4),
 ('zsa', 4),
 ('yves', 4),
 ('yoshi', 4)]